In [1]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV9Base
from tscglue import utils, data_loader
import polars as pl
from sklearn.metrics import log_loss, roc_auc_score


In [2]:
dataset = "Worms"
dataset = "BirdChicken"
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(20, 1, 512) (20,) (20, 1, 512) (20,)


In [3]:
seed = 243

In [4]:
m = LokyStackerV9Base(n_repetitions=1, random_state=seed, n_jobs=16, keep_features=True, verbose=2, stacking_models=['probability-et'])
m.fit(X_train, y_train)
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/dff7fe6e08bc4892
[0.00s] Saved X and y to disk in 0.00s
[0.17s] Computed QUANT features (20, 4500) (0.69 MB) in 0.1670s
[0.21s] Starting repetition 0
[1.65s] Computed MultiRocket features (20, 49728) (7.59 MB) in 1.4341s
[1.79s] Computed Hydra features (20, 6144) (0.94 MB) in 0.1373s
[3.31s] Computed RDST features (20, 29679) (4.53 MB) in 1.5201s
[3.31s] Starting training with 16 workers for 40 models


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [37868] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


[5.84s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2612s for f-0/r-0 (2.62 MB)
[5.88s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2179s for f-1/r-0 (2.62 MB)
[5.97s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2570s for f-2/r-0 (2.62 MB)
[5.99s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2091s for f-3/r-0 (2.62 MB)
[6.05s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2063s for f-4/r-0 (2.62 MB)
[6.07s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1778s for f-5/r-0 (2.62 MB)
[6.16s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2494s for f-6/r-0 (2.62 MB)
[6.17s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1761s for f-8/r-0 (2.62 MB)
[6.18s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2052s for f-7/r-0 (2.62 MB)
[6.24s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1849s for f-9/r-0 (2.62 MB)
[6.24s] Completed training for model multirockethydra-bestk-p-ridgecv_r0
[6.27s] OOF acc for model multirockethydra-bestk-p-ridgecv_r0: 0.9
[6.51s] Tr

In [5]:
y_prob_pred = m.predict_proba(X_test)
classes = list(m.classes_)
print(f"Log-loss: {log_loss(y_test, y_prob_pred, labels=classes):.4f}")
if y_prob_pred.shape[1] == 2:
    auc = roc_auc_score(y_test, y_prob_pred[:, 1])
else:
    auc = roc_auc_score(y_test, y_prob_pred, multi_class='ovr', labels=classes)
print(f"AUC (OvR): {auc:.4f}")

[0.00s] Starting prediction
[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/dff7fe6e08bc4892
[0.44s] Computed QUANT features (20, 4500) (0.34 MB) in 0.442807s
[0.50s] Computed repetition 0 MultiRocket features (20, 49728) (3.79 MB) in 0.058570s
[1.19s] Computed repetition 0 Hydra features (20, 6144) (0.47 MB) in 0.687862s
[1.60s] Computed repetition 0 RDST features (20, 29679) (4.53 MB) in 0.407659s
[1.63s] Computed and saved features for prediction
[1.63s] Starting prediction with 16 workers for 40 first-level models
[5.36s] Predicted rdst-p-ridgecv_r0 in 0.0097s
[5.37s] Predicted rdst-p-ridgecv_r0 in 0.0107s
[5.37s] Predicted rdst-p-ridgecv_r0 in 0.0110s
[5.37s] Predicted rdst-p-ridgecv_r0 in 0.0114s
[5.37s] Predicted rdst-p-ridgecv_r0 in 0.0144s
[5.37s] Predicted rdst-p-ridgecv_r0 in 0.0146s
[5.38s] Predicted rdst-p-ridgecv_r0 in 0.0095s
[5.38s] Predicted rdst-p-ridgecv_r0 in 0.0097s
[5.38s] Predicted rdst-p-ridgecv_r0 in 0.0105s
[5.38s] Predicted rdst-p-ridgecv_r0

In [6]:
m = LokyStackerV9Base(n_repetitions=5, random_state=seed, n_jobs=16, keep_features=True, verbose=2, stacking_models=['probability-et'])
m.fit(X_train, y_train)
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/b384fa7867064124
[0.00s] Saved X and y to disk in 0.00s
[0.18s] Computed QUANT features (20, 4500) (0.69 MB) in 0.1686s
[0.22s] Starting repetition 0
[0.95s] Computed MultiRocket features (20, 49728) (7.59 MB) in 0.7327s
[1.52s] Computed Hydra features (20, 6144) (0.94 MB) in 0.5702s
[2.97s] Computed RDST features (20, 29586) (4.51 MB) in 1.4458s
[2.97s] Starting training with 16 workers for 200 models
[7.03s] Trained rstsf_r0 in 1.9577s for f-0/r-0 (0.28 MB)
[7.34s] Trained rstsf_r0 in 2.0409s for f-1/r-0 (0.29 MB)
[7.41s] Trained rstsf_r0 in 2.0126s for f-3/r-0 (0.29 MB)
[7.50s] Trained rstsf_r0 in 2.0735s for f-4/r-0 (0.28 MB)
[7.53s] Trained rstsf_r0 in 2.0590s for f-7/r-0 (0.29 MB)
[7.58s] Trained rstsf_r0 in 2.1251s for f-5/r-0 (0.28 MB)
[7.63s] Trained rstsf_r0 in 2.1003s for f-8/r-0 (0.30 MB)
[7.65s] Trained rstsf_r0 in 1.9924s for f-9/r-0 (0.29 MB)
[7.82s] Trained rstsf_r0 in 2.1256s for f-10/r-0 (0.29 MB)
[7.86

/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [37745] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


[11.63s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2231s for f-21/r-0 (2.62 MB)
[11.65s] Trained rstsf_r0 in 1.4647s for f-45/r-0 (0.30 MB)
[11.67s] Trained rstsf_r0 in 1.5808s for f-44/r-0 (0.28 MB)
[11.68s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1929s for f-26/r-0 (2.62 MB)
[11.70s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2114s for f-27/r-0 (2.62 MB)
[11.73s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2147s for f-28/r-0 (2.62 MB)
[11.74s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2822s for f-25/r-0 (2.62 MB)
[11.77s] Trained rstsf_r0 in 1.4836s for f-46/r-0 (0.29 MB)
[11.77s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1990s for f-29/r-0 (2.62 MB)
[11.78s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2052s for f-30/r-0 (2.62 MB)
[11.79s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1939s for f-31/r-0 (2.62 MB)
[11.82s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2086s for f-32/r-0 (2.62 MB)
[11.82s] Trained multirockethydra-bes

/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [37745] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


[11.83s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2022s for f-34/r-0 (2.62 MB)
[11.87s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1962s for f-37/r-0 (2.62 MB)
[11.90s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2502s for f-35/r-0 (2.62 MB)
[11.93s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2201s for f-38/r-0 (2.62 MB)
[11.93s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1596s for f-42/r-0 (2.62 MB)
[11.97s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2310s for f-40/r-0 (2.62 MB)
[12.00s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2069s for f-44/r-0 (2.62 MB)
[12.00s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2137s for f-43/r-0 (2.62 MB)
[12.01s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.2767s for f-39/r-0 (2.62 MB)
[12.01s] Trained rstsf_r0 in 1.5267s for f-49/r-0 (0.28 MB)
[12.01s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.1879s for f-46/r-0 (2.62 MB)
[12.02s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.3389s for f-

In [7]:
y_prob_pred = m.predict_proba(X_test)
classes = list(m.classes_)
print(f"Log-loss: {log_loss(y_test, y_prob_pred, labels=classes):.4f}")
if y_prob_pred.shape[1] == 2:
    auc = roc_auc_score(y_test, y_prob_pred[:, 1])
else:
    auc = roc_auc_score(y_test, y_prob_pred, multi_class='ovr', labels=classes)
print(f"AUC (OvR): {auc:.4f}")

[0.00s] Starting prediction
[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/b384fa7867064124
[0.53s] Computed QUANT features (20, 4500) (0.34 MB) in 0.526951s
[0.65s] Computed repetition 0 MultiRocket features (20, 49728) (3.79 MB) in 0.125583s
[1.30s] Computed repetition 0 Hydra features (20, 6144) (0.47 MB) in 0.649900s
[1.95s] Computed repetition 0 RDST features (20, 29586) (4.51 MB) in 0.647397s
[1.99s] Computed and saved features for prediction
[1.99s] Starting prediction with 16 workers for 200 first-level models
[22.46s] Predicted rstsf_r0 in 16.9957s
[25.39s] Predicted rstsf_r0 in 19.9231s
[25.53s] Predicted rstsf_r0 in 20.0718s
[25.56s] Predicted rstsf_r0 in 20.0968s
[25.58s] Predicted rstsf_r0 in 20.1095s
[25.94s] Predicted rstsf_r0 in 20.4701s
[26.07s] Predicted rstsf_r0 in 20.6043s
[26.24s] Predicted rstsf_r0 in 20.7767s
[28.39s] Predicted rstsf_r0 in 22.9339s
[28.60s] Predicted rstsf_r0 in 23.1301s
[29.31s] Predicted rstsf_r0 in 23.8343s
[30.01s] Predicte